In [1]:
import os
import pandas as pd
import numpy as np

root_path = r"../dataset"
audit_summary = []

def get_sample_rows(df):
    n = len(df)
    start = df.head(2)
    mid = df.iloc[n//2:n//2+2] if n > 4 else pd.DataFrame()
    end = df.tail(2) if n > 2 else pd.DataFrame()
    return start, mid, end

def inspect_file(path):
    print("\n" + "="*120)
    print("FILE:", path)
    print("="*120)

    try:
        if path.endswith(".csv"):
            df = pd.read_csv(path, low_memory=False)
        elif path.endswith(".xlsx"):
            df = pd.read_excel(path)
        else:
            return
        
        rows, cols = df.shape
        print("Shape:", df.shape)
        print("\nColumns:")
        print(list(df.columns))

        print("\nDtypes:")
        print(df.dtypes)

        print("\nMissing % per column:")
        print((df.isna().mean()*100).round(2))

        print("\nDuplicate rows:", df.duplicated().sum())

        num_df = df.select_dtypes(include=np.number)
        if not num_df.empty:
            print("\nNumeric Summary:")
            print(num_df.describe())

        start, mid, end = get_sample_rows(df)

        print("\n--- First 2 Rows ---")
        print(start)

        print("\n--- Middle 2 Rows ---")
        print(mid)

        print("\n--- Last 2 Rows ---")
        print(end)

        audit_summary.append({
            "file": path,
            "rows": rows,
            "cols": cols,
            "numeric_cols": num_df.shape[1],
            "duplicate_rows": df.duplicated().sum(),
            "size_mb": round(os.path.getsize(path)/1e6,2)
        })

    except Exception as e:
        print("Error:", e)


for root, dirs, files in os.walk(root_path):
    
    # Skip preview junk files
    files = [f for f in files if "_preview" not in f]
    
    # SPECIAL RULE: prices of energy → inspect only one file per year
    if "prices of energy" in root.lower():
        
        # Only process first valid file in that year folder
        valid_files = [f for f in files if f.endswith((".csv", ".xlsx"))]
        if valid_files:
            path = os.path.join(root, valid_files[0])
            inspect_file(path)
        
        continue

    # ALL OTHER FOLDERS → inspect all files
    for file in files:
        if file.endswith((".csv", ".xlsx")):
            path = os.path.join(root, file)
            inspect_file(path)


summary_df = pd.DataFrame(audit_summary)
summary_df.to_csv("dataset_audit_final_research.csv", index=False)

print("\n\nFINAL AUDIT FILE CREATED: dataset_audit_final_research.csv")
print(summary_df)


FILE: ../dataset\CEA Data\CO2_DatabaseVersion_19_2022_23.xlsx
Shape: (2039, 27)

Columns:
['S_NO', 'NAME', 'UNIT_NO', 'DT_ COMM', 'CAPACITY MW AS ON 31/03/2023', 'TYPE', 'STATE', 'SECTOR', 'SYSTEM', 'Fuel1', 'Fuel2', 'Gross Gen (MU or GWh)', 'Auxilary Cons.(%)', 'NET GENERATION(MU)', 'Fuel 1 Consumption (if coal: only national coal)', 'Fuel 1 Consumption (only for imported coal)', 'Fuel 2 Consumption', 'GCV(kCal/Kg) Fuel 1                   (if coal: only national coal)', 'GCV(kCal/Kg) Fuel 1                   (only for imported coal)', 'GCV(kCal/Kg) Fuel 2', 'CO2(ton)- FUEL-1', 'CO2(ton)- FUEL-2', 'CO2 FUEL2 %', 'TOTAL CO2(ton)', 'Emission factor (ton/MWh)', 'in Operating Margin', 'In Build Margin']

Dtypes:
S_NO                                                                   float64
NAME                                                                    object
UNIT_NO                                                                 object
DT_ COMM                                   